## Pomiar i preprocessing sygnału EMG
Procedura analizy sygnału EMG: celem jest dokonanie pomiaru sygnału elektrycznego aktywności mięśni szkieletowych (biceps) - elektromiografia EMG, oraz analizy sygnału - określenie magnitudy aktywacji mięśnia, widma sygnału EMG, stopnia zmęczenia.

#### 1. Rozmieszczenie elektrod do pomiaru sygnału EMG 
 - użyj system pomiarowy jak do pomiaru EKG;
 - dwie elektrody pomiarowe powinny znajdować się pomiędzy strefą unerwienia a przyczepem ścięgna mięśnia, wzdłuż środkowej linii podłużnej mięśnia. Powinny być oddalone od siebie o 2–4 cm i umieszczone nad brzuchem mięśnia, aby ograniczyć zakłócenia pochodzące od innych mięśni. Elektroda odniesienia powinna znajdować się na części ciała pokrytej kością, charakteryzującej się bardzo małą aktywnością mięśniową;
 - biała elektroda powinna znajdować się w środkowej części brzucha mięśnia;
 - czarna elektroda jest elektrodą odniesienia (nadgarstek)
 - przydatne informacje: http://seniam.org/; https://docs.google.com/document/d/e/2PACX-1vRH3qfvMvmHBdHhg3sFQKTwo1-blg8luLXSQRhsIhEgwFj3QmWGStUhvslrVuVzP-aQ3T0YNs-lUsOz/pub

#### 2. Procedura pomiaru sygnału EMG 
a) Eksperyment 1: określenie magnitudy aktywacji mięśnia - Maximum Voluntary Contraction (MVC)
 - chwyć stół ręką, utrzymując ramię w pozycji pionowej i łokieć pod kątem 90 stopni. Podnieś stół z całej siły przez około 5 sekund. Staraj się nie zwiększać siły stopniowo, ale maksymalnie napnij mięśnie w ciągu jednej sekundy. Staraj się jak najmniej poruszać kablami (np. nie ocieraj ramienia o bok ciała);
 - zrób co najmniej 60-sekundową przerwę i powtórz tę samą procedurę dwa razy, aby uzyskać trzy oddzielne zestawy danych MVC;

b) Eksperyment 2: określenie stopnia aktywacji mięśnia - Relative Muscle Acivation
 - spróbuj znaleźć w domu 3 ciężarki, które po podniesieniu wyraźnie różnią się od siebie. Najlepiej, żeby te ciężarki miały około 25%, 50% i 75% siły, którą wygenerowałeś izometrycznie w eksperymencie 1;
 - dokonaj rejestracji sygnału dla 3 różnych obciążeń w 3 różnych plikach

c) Eksperyment 3: określenie stopnia zmęczenia mięśnia 
 - zmierz aktywność mięśnia dwugłowego ramienia podczas maksymalnego skurczu (tak mocno, jak to możliwe!) przez pełne 10 sekund (pełne 10 sekund!);
 - zrób co najmniej 60-sekundową przerwę. Powtórz procedurę jeszcze dwa razy, aby uzyskać trzy zestawy danych dotyczących zmęczenia;

#### 3. Import bibliotek

In [87]:
%matplotlib inline
import matplotlib.pyplot as plt
import emg_lib as el
import emg_functions as ef
import numpy as np
import scipy as sp

#### 4. Wczytanie danych pomiarowych z pliku

In [90]:
#połączenie 3 plików odpowiednio MVC#, Weight#, Fatigue#
weights, mvc, fatigue = ef.import_data(',')

In [92]:
print(mvc.emg)

0        318
1        319
2        318
3        319
4        319
        ... 
37309    312
37310    312
37311    312
37312    312
37313    311
Name: emg, Length: 37314, dtype: int64


In [94]:
print(mvc.t)

0          719
1          720
2          721
3          722
4          723
         ...  
37309    38182
37310    38183
37311    38184
37312    38186
37313    38187
Name: t, Length: 37314, dtype: int64


#### 5. Wizualizacja wyników pomiarów

In [97]:
#MVC
plt.plot(mvc.t, mvc.emg)
plt.title('Maximum Voluntary Contraction')
plt.ylabel('MVC / a. u.')
plt.xlabel('t / s')

Text(0.5, 0, 't / s')

In [99]:
#Weights
plt.plot(weights.t, weights.emg)
plt.title('Relative Muscle Activation')
plt.ylabel('Weights / a. u.')
plt.xlabel('t / s')

Text(0.5, 23.52222222222222, 't / s')

In [101]:
#Fatigue
plt.plot(fatigue.t, fatigue.emg)
plt.title('Fatigue')
plt.ylabel('Fatigue / a. u.')
plt.xlabel('t / s')

Text(0.5, 23.52222222222222, 't / s')

#### 6. Usunięcie offsetu

In [104]:
mvc_correctmean = ef.remove_mean(mvc.t, mvc.emg)
weights_correctmean = ef.remove_mean(weights.t, weights.emg)
fatigue_correctmean = ef.remove_mean(fatigue.t, fatigue.emg)

#### 7. Filtracja + usuwanie wartosci ujemnych + obwiednia z wszystkich sygnałów
Surowy sygnał EMG przypomina szum. Aby wyciągnąć z niego użyteczne informacje o sile skurczu, stosujemy potrójny proces:
1. **Filtracja pasmowo-przepustowa** (np. 20-450 Hz) – usuwa artefakty ruchowe i szumy sprzętowe.
2. **Prostowanie (Rectification)** – zamiana wartości ujemnych na dodatnie (wartość bezwzględna).
3. **Wygładzanie (Envelope)** – tworzy obwiednię sygnału za pomocą filtru dolnoprzepustowego lub algorytmu RMS.

In [107]:
# Zastosowanie kompleksowej filtracji: czas, wyśrodkowany sygnał, krok, fs=1000Hz, low=20Hz, high=450Hz
mvc_filt, mvc_env = ef.filteremg(mvc.t, mvc_correctmean, 0.5, 1000, 20, 450)
weights_filt, weights_env = ef.filteremg(weights.t, weights_correctmean, 1, 1000, 20, 450)
fatigue_filt, fatigue_env = ef.filteremg(fatigue.t, fatigue_correctmean, 1, 1000, 20, 450)

#### 8. Wyznaczanie przedziałów pracy mięśni i Normalizacja

Aby porównywać wyniki między pacjentami lub sesjami, stosuje się **normalizację do MVC**. Oznacza to, że każdy badany skurcz wyrażamy jako procent maksymalnej siły, jaką dana osoba jest w stanie wygenerować. Najpierw jednak algorytm musi wykryć, w których momentach (indeksach) występował faktyczny skurcz (tzw. *burst*).

In [110]:
%matplotlib widget
#%matplotlib notebook
#%matplotlib qt
# Wyznaczanie początków i końców skurczów (bursts)
mvc_start, mvc_end, weights_start, weights_end, fatigue_start, fatigue_end = el.get_bursts(mvc_filt, weights_filt, fatigue_filt)

# Obliczenie średniej amplitudy dla każdego z 3 skurczów MVC
mean_mvc1 = np.mean(mvc_env[mvc_start[0]:mvc_end[0]])
mean_mvc2 = np.mean(mvc_env[mvc_start[1]:mvc_end[1]])
mean_mvc3 = np.mean(mvc_env[mvc_start[2]:mvc_end[2]])

# Uśrednienie wyników z 3 prób MVC
mean_mvc = (mean_mvc1 + mean_mvc2 + mean_mvc3) / 3

# Normalizacja prób z obciążeniem (% MVC)
load1 = (np.mean(weights_env[weights_start[0]:weights_end[0]])) / mean_mvc * 100
load2 = (np.mean(weights_env[weights_start[1]:weights_end[1]])) / mean_mvc * 100
load3 = (np.mean(weights_env[weights_start[2]:weights_end[2]])) / mean_mvc * 100

print(f"Obciążenie 1: {load1:.2f}% MVC")
print(f"Obciążenie 2: {load2:.2f}% MVC")
print(f"Obciążenie 3: {load3:.2f}% MVC")

RuntimeError: 'widget is not a recognised GUI loop or backend name

#### 9. Analiza zmęczenia w domenie częstotliwości (Widmo Mocy)

Kiedy mięsień się męczy, zmienia się jego fizjologia (m.in. gromadzi się kwas mlekowy, zwalnia szybkość przewodzenia). Objawia się to **przesunięciem widma częstotliwościowego w stronę niższych wartości**. Zbadamy to, analizując transformatę Fouriera sygnału zmęczeniowego w trzech różnych etapach jego trwania.

In [ ]:
# Obliczenie widma mocy sygnału zmęczeniowego (dla 3 etapów) przy fs = 1000 Hz
power1, frequencies1 = el.get_power(fatigue_filt[fatigue_start[0]:fatigue_end[0]], 1000)
power2, frequencies2 = el.get_power(fatigue_filt[fatigue_start[1]:fatigue_end[1]], 1000)
power3, frequencies3 = el.get_power(fatigue_filt[fatigue_start[2]:fatigue_end[2]], 1000)

plt.figure(figsize=(10, 5))
plt.plot(frequencies1, power1, label='Etap 1', color='blue')
plt.plot(frequencies2, power2, label='Etap 2', color='red')
plt.plot(frequencies3, power3, label='Etap 3', color='green')
plt.title('Widmo mocy sygnału zmęczeniowego (surowe)')
plt.xlabel('Częstotliwość [Hz]')
plt.ylabel('Moc')
plt.legend()
plt.show()

#### 10. Wygładzanie widma i obliczanie Mediany Częstotliwości (MDF)

Surowe widmo jest poszarpane, dlatego najpierw wygładzamy je filtrem dolnoprzepustowym. 
Głównym wskaźnikiem zmęczenia jest **Mediana Częstotliwości (Median Frequency - MDF)**. Jest to taka częstotliwość, która dzieli pole pod wykresem widma na dwie równe połowy. W miarę narastania zmęczenia, wartość MDF powinna spadać.

In [ ]:
# Filtracja widma (wygładzanie wykresów z poprzedniego kroku)
low_pass = 5 / (1000 / 2) # Znormalizowana częstotliwość odcięcia (fs = 1000)
b2, a2 = sp.signal.butter(4, low_pass, btype='lowpass')

pow_filt1 = sp.signal.filtfilt(b2, a2, power1)
pow_filt2 = sp.signal.filtfilt(b2, a2, power2)
pow_filt3 = sp.signal.filtfilt(b2, a2, power3)

# Opcjonalna własna funkcja do całkowania numerycznego
'''
def pseudo_cumtrapz(y, x):
    dx = np.diff(x)
    avg_y = (y[:-1] + y[1:]) / 2
    cum_area = np.concatenate([[0], np.cumsum(avg_y * dx)])
    return cum_area
'''

# Obliczanie pola pod wykresem (całka) i wyznaczanie mediany częstotliwości
area_freq1 = sp.integrate.cumtrapz(pow_filt1, frequencies1, initial=0)
median_freq1 = frequencies1[np.where(area_freq1 >= area_freq1[-1] / 2)[0][0]]

area_freq2 = sp.integrate.cumtrapz(pow_filt2, frequencies2, initial=0)
median_freq2 = frequencies2[np.where(area_freq2 >= area_freq2[-1] / 2)[0][0]]

area_freq3 = sp.integrate.cumtrapz(pow_filt3, frequencies3, initial=0)
median_freq3 = frequencies3[np.where(area_freq3 >= area_freq3[-1] / 2)[0][0]]

median_freq = [median_freq1, median_freq2, median_freq3]

# Wykres spadku mediany częstotliwości
plt.figure(figsize=(8, 5))
plt.plot(['Etap 1', 'Etap 2', 'Etap 3'], median_freq, marker='X', markersize=10, linestyle='-', color='purple')
plt.title('Zmiana Mediany Częstotliwości (Spadek = Zmęczenie)')
plt.ylabel('Mediana Częstotliwości [Hz]')
plt.grid(True, linestyle='--', alpha=0.6)
plt.show()